# 스킵 연결로 깊게 쌓기: DeepGCN 과 GCNII

층을 깊게 쌓으면 노드 표현이 서로 비슷해져요. 그것을 막는 가장 단순한 방법이 '원래 값을 더해 주기' 예요. 여기서는 DeepGCN 의 잔차 연결과 GCNII 의 초기 잔차, 항등 사상을 numpy 로 직접 짜 봐요. 5주차는 실습 노트북이 없어서 슬라이드의 식(p.37, p.42-43)과 기출 계산 모양으로 만들었어요. 마지막 실습만 슬라이드 p.44 의 GCN2Conv 코드를 그대로 써서 Colab 에서 열려요.

**하는 법**
1. 위 메뉴 **런타임 > 런타임 유형 변경** 은 CPU 그대로 두어도 돼요.
2. 문제마다 **내 코드 칸**을 채우고 실행(Shift+Enter)한 뒤, 바로 아래 **채점 칸**을 실행해요.
3. `통과!` 가 나오면 성공, 빨간 `AssertionError` 가 나오면 마지막 줄의 한국어 안내를 읽고 고쳐요.
4. 막히면 **힌트**를 한 단계씩 펼쳐 보고, 그래도 안 되면 **정답 보기**를 펼쳐요.

## 준비: 필요한 도구 설치 (처음 한 번만 실행)

In [ ]:
!pip install -q torch_geometric

## 1. PyG 의 GCN2Conv 층 쌓기  (직접 짜기)

슬라이드 p.44 의 GCNII 모델에서 층을 만드는 부분을 그대로 짜 봐요.

- `from torch_geometric.nn import GCN2Conv` 가 슬라이드 맨 윗줄이에요. PyG 가 GCNII 층을 이미 만들어 두었어요.
- `torch.nn.ModuleList()` 는 층을 담는 특별한 리스트예요. 보통 리스트에 담으면 PyTorch 가 파라미터를 못 찾아요.
- `GCN2Conv(hidden_channels, alpha, theta, layer, shared_weights, normalize)` 순서예요. 슬라이드는 $\alpha = 0.1$, $\theta = 0.5$ 를 썼어요.
- 네 번째 자리 `layer + 1` 이 층 번호예요. 1 부터 세기 때문에 $+1$ 을 해요. GCNII 는 이 번호로 $\beta_\ell = \ln(\theta/\ell + 1)$ 을 정해요.
- 이 실습은 PyTorch 가 필요해서 Colab 에서 열려요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 은닉 칸 수와 층 수이고 출력은 층들이 담긴 ModuleList 예요.
2. 반복마다 새 층을 하나 만들어 리스트에 붙이는 구조예요.
3. 층 번호가 0 이 아니라 1 부터라는 점이 함정이에요.
4. `len(convs)` 가 num_layers 와 같은지로 확인할 수 있어요.

슈도코드

```text
층을 담을 빈 ModuleList 를 만든다
0 부터 num_layers - 1 까지 반복한다
    GCN2Conv 를 하나 만들어 리스트에 붙인다
리스트를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

torch.nn.ModuleList() 로 시작해요

</details>

<details><summary>힌트 2</summary>

for layer in range(num_layers) 로 돌아요

</details>

<details><summary>힌트 3</summary>

네 번째 인자는 layer 가 아니라 layer + 1 이에요

</details>

원본: L5 슬라이드 p.44 (GCN2Conv 예시 코드)


In [ ]:
import torch
from torch_geometric.nn import GCN2Conv

# 1. hidden_channels 칸짜리 층을 num_layers 개 담을 빈 ModuleList 를 만들어요
# 2. layer 를 0 부터 num_layers - 1 까지 돌면서
#    GCN2Conv(hidden_channels, 0.1, 0.5, layer + 1, shared_weights, normalize=False) 를 append 해요
# 3. convs 라는 이름의 ModuleList 를 돌려주는 함수 make_convs(...) 를 완성해요
def make_convs(hidden_channels, num_layers, shared_weights=True):
    ...


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
assert "make_convs" in globals(), "make_convs 함수를 만들어야 해요"
convs = make_convs(16, 4)
assert convs is not None, "return 을 빠뜨렸어요"
assert isinstance(convs, torch.nn.ModuleList), "보통 리스트가 아니라 torch.nn.ModuleList 여야 해요"
assert len(convs) == 4, f"층이 4 개여야 해요. 지금은 {len(convs)} 개예요"
assert all(isinstance(c, GCN2Conv) for c in convs), "리스트 안은 전부 GCN2Conv 여야 해요"
assert convs[0].channels == 16, f"은닉 칸 수가 16 이어야 해요. 지금은 {convs[0].channels} 이에요"
assert abs(convs[0].alpha - 0.1) < 1e-9, "슬라이드는 alpha = 0.1 을 썼어요"
assert len(make_convs(8, 2)) == 2, "층 수를 바꿔도 맞아야 해요"
b0 = convs[0].beta
b3 = convs[3].beta
assert b0 > b3, "층 번호를 layer + 1 로 넘겨야 깊은 층일수록 beta 가 작아져요"
print("통과! 슬라이드 p.44 의 GCNII 클래스가 바로 이 조각을 `__init__` 안에서 써요. 나머지는 Linear 두 개와 Dropout 이에요.")

<details><summary>정답 보기</summary>

```python
import torch
from torch_geometric.nn import GCN2Conv


def make_convs(hidden_channels, num_layers, shared_weights=True):
    convs = torch.nn.ModuleList()
    for layer in range(num_layers):
        convs.append(
            GCN2Conv(hidden_channels, 0.1, 0.5, layer + 1, shared_weights, normalize=False))
    return convs

```

</details>